# LLaMA 3.1 on SageMaker

This notebook shows how to:

- Deploy a Hugging Face LLaMA 3.1 model on Amazon SageMaker  
- Run a quick test inference  
- **Always tear down the endpoint** to avoid idle GPU charges


## 0) Parameters

- Region defaults to the current AWS session, or `ap-southeast-2` if none is set  
- Endpoint resources (endpoint, config, model) will be **auto-cleaned** in a `finally` block


In [1]:
import os, json, time, boto3, botocore, sagemaker
from sagemaker.huggingface import HuggingFaceModel

REGION = os.environ.get("AWS_REGION") or boto3.Session().region_name or "ap-southeast-2"
ROLE = sagemaker.get_execution_role()  # works on Notebook Instance

MODEL_ID = "meta-llama/Llama-2-7b-hf"
ENDPOINT_NAME = "llama31-8b-endpoint"
INSTANCE_TYPE = "ml.g5.2xlarge"   # GPU cost, keep usage short!
AUTO_TIMEOUT_MIN = 20

IMAGE_URI = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04"

print("Region:", REGION)
print("Role:", ROLE)
print("Model:", MODEL_ID)
print("Endpoint:", ENDPOINT_NAME)
print("Image URI:", IMAGE_URI)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Region: ap-southeast-2
Role: arn:aws:iam::575935529773:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole
Model: meta-llama/Llama-2-7b-hf
Endpoint: llama31-8b-endpoint
Image URI: 763104351884.dkr.ecr.ap-southeast-2.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04


## 1) Clients & helpers (cleanup + wait)
Setup SageMaker clients and define small helper functions to wait for endpoints and clean them up safely.

In [2]:
sm = boto3.client("sagemaker", region_name=REGION)
rt = boto3.client("sagemaker-runtime", region_name=REGION)

def safe_call(fn, **kw):
    try:
        return fn(**kw)
    except botocore.exceptions.ClientError as e:
        code = e.response.get("Error", {}).get("Code")
        if code in {"ValidationException", "ResourceNotFound"}:
            return None
        raise

def kill(name: str):
    """Best-effort, idempotent teardown in correct order."""
    safe_call(sm.delete_endpoint, EndpointName=name)
    safe_call(sm.delete_endpoint_config, EndpointConfigName=name)
    safe_call(sm.delete_model, ModelName=name)

def wait_inservice(name: str):
    last = None
    start = time.time()
    while True:
        d = sm.describe_endpoint(EndpointName=name)
        st = d["EndpointStatus"]
        if st != last:
            print("Endpoint status:", st)
            last = st
        if st in ("InService", "Failed"):
            if st == "Failed":
                print("FailureReason:\n", d.get("FailureReason"))
            return st
        if (time.time() - start) > (AUTO_TIMEOUT_MIN * 60):
            print(f"Timeout waiting for InService (> {AUTO_TIMEOUT_MIN} min). Aborting.")
            return "TimedOut"
        time.sleep(10)

## 2) Model Definition

Define the Hugging Face model container, environment variables, and runtime configuration for deployment.

In [3]:
# ---- base environment ----
env = {
    "HF_MODEL_ID": MODEL_ID,
    "HF_TASK": "text-generation",
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "MAX_INPUT_LENGTH": "4096",   # TODO: edit later
    "MAX_TOTAL_TOKENS": "4096",   # TODO: edit later
}

# ---- fetch HF token from Secrets Manager ----
SECRET_NAME = "AmazonSageMaker-johannegg"
sm_secrets = boto3.client("secretsmanager", region_name=REGION)
val = sm_secrets.get_secret_value(SecretId=SECRET_NAME)
HF_TOKEN = json.loads(val["SecretString"])["password"]

env["HF_TOKEN"] = HF_TOKEN
print("Loaded HF token from Secrets Manager ✔︎")

# ---- define model ----
hf_model = HuggingFaceModel(
    image_uri=IMAGE_URI,
    role=ROLE,
    env=env,
)

Loaded HF token from Secrets Manager ✔︎


In [9]:
import boto3, json
print(boto3.client("sts").get_caller_identity())

{'UserId': 'AROAYMGDOYMWW5XXFRQUU:SageMaker', 'Account': '575935529773', 'Arn': 'arn:aws:sts::575935529773:assumed-role/AmazonSageMakerServiceCatalogProductsUseRole/SageMaker', 'ResponseMetadata': {'RequestId': '7b32a3d9-6a09-4f48-be3c-91d57a737a29', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '7b32a3d9-6a09-4f48-be3c-91d57a737a29', 'x-amz-sts-extended-request-id': 'MTphcC1zb3V0aGVhc3QtMjoxNzU5NzE3ODg4MzQ4OlI6TzJBdzRLaUQ=', 'content-type': 'text/xml', 'content-length': '469', 'date': 'Mon, 06 Oct 2025 02:31:28 GMT'}, 'RetryAttempts': 0}}


## 3) Deploy → Test → Teardown

Deploy the model to a SageMaker endpoint, run a quick inference test, and always clean up resources to avoid costs.

In [4]:
# --- Deploy (non-blocking) ---
predictor = hf_model.deploy(
    endpoint_name=ENDPOINT_NAME,
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    container_startup_health_check_timeout=1800,
    wait=False,  # don't block the notebook
)

# --- Wait until InService ---
status = wait_inservice(ENDPOINT_NAME)
if status != "InService":
    raise RuntimeError(f"Endpoint not ready (status={status}).")

# --- Test inference(s) ---
payload = {
    "inputs": "Give me 3 one-sentence risks of using brain-computer interfaces.",
    "parameters": {"temperature": 0.7, "max_new_tokens": 120}
}
resp = rt.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(payload),
)
print("Inference:", json.loads(resp["Body"].read()))
print("Endpoint left running:", ENDPOINT_NAME)


Endpoint status: Creating
Timeout waiting for InService (> 20 min). Aborting.


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:13                                                                                   │
│                                                                                                  │
│   10 # --- Wait until InService ---                                                              │
│   11 status = wait_inservice(ENDPOINT_NAME)                                                      │
│   12 if status != "InService":                                                                   │
│ ❱ 13 │   raise RuntimeError(f"Endpoint not ready (status={status}).")                            │
│   14                                                                                             │
│   15 # --- Test inference(s) ---                                                                 │
│   16 payload = {                                                                                 │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
RuntimeError: Endpoint not ready (status=TimedOut).

### 3.1 Cleanup

In [5]:
print("Tearing down:", ENDPOINT_NAME)
kill(ENDPOINT_NAME)

Tearing down: llama31-8b-endpoint


### 3.2 Check status

In [ ]:
sm = boto3.client("sagemaker", region_name=REGION)
try:
    st = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
    print(ENDPOINT_NAME, "->", st)
except sm.exceptions.ClientError:
    print("Endpoint not found (deleted).")